# Schema Validation 1 results and analysis

Inspect successful assessments, summarize schema coverage signals, review candidate-field evidence, and calculate API spend from recorded token usage.

Costs use [OpenAI's Standard and Flex processing prices](https://developers.openai.com/api/docs/pricing) for GPT-5.5 with less than 272K context, retrieved August 25, 2026. Only successful calls in the results JSONL are included. Candidate names are shown exactly as proposed and are not consolidated across aliases.

In [ ]:
import json
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Image, Markdown, display

from schema_development.paths import ROOT


## Configuration

In [ ]:
RESULTS_PATH = ROOT / "artifacts/validation1/results.jsonl"
ERRORS_PATH = ROOT / "runs/validation1/errors.jsonl"
SNAPSHOTS_DIR = ROOT / "data/source/heldout/snapshots"

# USD per 1 million tokens, context under 272K.
PRICES_USD_PER_1M = {
    ("gpt-5.5", "default"): {
        "input": 5.00,
        "cached_input": 0.50,
        "output": 30.00,
    },
    ("gpt-5.5", "flex"): {
        "input": 2.50,
        "cached_input": 0.25,
        "output": 15.00,
    },
}


In [ ]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    """Load JSON objects from a JSONL file.

    Parameters
    ----------
    path : Path
        JSONL file to read.

    Returns
    -------
    list[dict[str, Any]]
        Parsed records, or an empty list when the file does not exist.

    Raises
    ------
    ValueError
        If a non-empty line is not valid JSON.
    """
    if not path.exists():
        return []

    records = []
    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path} at line {line_number}."
                ) from exc
    return records


def calculate_cost_usd(record: dict[str, Any]) -> float:
    """Calculate one successful call's token cost in US dollars.

    Parameters
    ----------
    record : dict[str, Any]
        Successful validation result containing model and usage data.

    Returns
    -------
    float
        Estimated API cost using the recorded service-tier prices.

    Raises
    ------
    ValueError
        If usage is missing or cached tokens exceed input tokens.
    KeyError
        If pricing is not configured for the model and service tier.
    """
    usage = record.get("usage")
    if not isinstance(usage, dict):
        raise ValueError(f"Missing usage for {record.get('snapshot_file_name')}.")

    raw_response = record.get("raw_response") or {}
    service_tier = raw_response.get("service_tier") or (
        record.get("request_config") or {}
    ).get("service_tier", "default")
    if service_tier == "auto":
        service_tier = "default"

    model = record["model"]
    prices = PRICES_USD_PER_1M[(model, service_tier)]
    input_tokens = int(usage["input_tokens"])
    cached_tokens = int(
        (usage.get("input_tokens_details") or {}).get("cached_tokens", 0)
    )
    output_tokens = int(usage["output_tokens"])
    if cached_tokens > input_tokens:
        raise ValueError("Cached input tokens cannot exceed input tokens.")

    cost = (input_tokens - cached_tokens) * prices["input"]
    cost += cached_tokens * prices["cached_input"]
    cost += output_tokens * prices["output"]
    return cost / 1_000_000


## Load results

In [ ]:
results = load_jsonl(RESULTS_PATH)
errors = load_jsonl(ERRORS_PATH)
if not results:
    raise ValueError(f"No successful results found at {RESULTS_PATH}.")

rows = []
for record in results:
    usage = record["usage"]
    parsed = record["parsed_output"]
    fit_counts = Counter(
        observation["fit_status"] for observation in parsed["observations"]
    )
    cached_tokens = (usage.get("input_tokens_details") or {}).get("cached_tokens", 0)
    service_tier = (record.get("raw_response") or {}).get("service_tier") or (
        record.get("request_config") or {}
    ).get("service_tier", "default")
    rows.append(
        {
            "snapshot_file_name": record["snapshot_file_name"],
            "source": record["source"],
            "artifact_type": record["artifact_type"],
            "model": record["model"],
            "service_tier": service_tier,
            "input_tokens": usage["input_tokens"],
            "cached_input_tokens": cached_tokens,
            "output_tokens": usage["output_tokens"],
            "total_tokens": usage["total_tokens"],
            "elapsed_seconds": record["elapsed_seconds"],
            "observations": len(parsed["observations"]),
            "covered": fit_counts["covered"],
            "weak_fit": fit_counts["weak_fit"],
            "no_fit": fit_counts["no_fit"],
            "out_of_scope": fit_counts["out_of_scope"],
            "uncertain": fit_counts["uncertain"],
            "candidate_fields": len(parsed["candidate_new_fields"]),
            "cost_usd": calculate_cost_usd(record),
        }
    )

results_df = (
    pd.DataFrame(rows)
    .sort_values(["source", "artifact_type", "snapshot_file_name"])
    .reset_index(drop=True)
)
records_by_name = {record["snapshot_file_name"]: record for record in results}

observation_rows = []
candidate_rows = []
candidate_evidence_rows = []
for record in results:
    parsed = record["parsed_output"]
    observation_by_id = {
        observation["observation_id"]: observation
        for observation in parsed["observations"]
    }
    record_context = {
        "snapshot_file_name": record["snapshot_file_name"],
        "source": record["source"],
        "artifact_type": record["artifact_type"],
    }
    for observation in parsed["observations"]:
        observation_rows.append(record_context | observation)
    for candidate in parsed["candidate_new_fields"]:
        supporting_observations = [
            observation_by_id[observation_id]
            for observation_id in candidate["supporting_observation_ids"]
        ]
        supporting_fit_counts = Counter(
            observation["fit_status"] for observation in supporting_observations
        )
        candidate_rows.append(
            record_context
            | candidate
            | {
                "weak_fit_support": supporting_fit_counts["weak_fit"],
                "no_fit_support": supporting_fit_counts["no_fit"],
            }
        )
        for observation in supporting_observations:
            candidate_evidence_rows.append(
                record_context
                | {
                    "proposed_name": candidate["proposed_name"],
                    "candidate_source_level": candidate["source_level"],
                    "definition": candidate["definition"],
                    "why_existing_fields_are_insufficient": candidate[
                        "why_existing_fields_are_insufficient"
                    ],
                    "operational_value": candidate["operational_value"],
                    "observation_id": observation["observation_id"],
                    "fit_status": observation["fit_status"],
                    "metadata_concept": observation["metadata_concept"],
                    "evidence_source": observation["evidence_source"],
                    "evidence": observation["evidence"],
                    "fit_rationale": observation["fit_rationale"],
                }
            )

observations_df = pd.DataFrame(observation_rows)
candidates_df = pd.DataFrame(candidate_rows)
candidate_evidence_df = pd.DataFrame(candidate_evidence_rows)
successful_files = set(records_by_name)
unresolved_errors = [
    error for error in errors if error.get("snapshot_file_name") not in successful_files
]

print(
    f"Loaded {len(results):,} successful calls, {len(errors):,} error attempts, "
    f"and {len(unresolved_errors):,} unresolved errors."
)


## API spend

In [ ]:
overall_cost = pd.DataFrame(
    {
        "successful_calls": [len(results_df)],
        "total_api_spend_usd": [results_df["cost_usd"].sum()],
        "average_cost_per_api_call_usd": [results_df["cost_usd"].mean()],
        "total_input_tokens": [results_df["input_tokens"].sum()],
        "total_cached_input_tokens": [results_df["cached_input_tokens"].sum()],
        "total_output_tokens": [results_df["output_tokens"].sum()],
    }
)
display(
    overall_cost.style.format(
        {
            "total_api_spend_usd": "${:,.4f}",
            "average_cost_per_api_call_usd": "${:,.4f}",
            "total_input_tokens": "{:,}",
            "total_cached_input_tokens": "{:,}",
            "total_output_tokens": "{:,}",
        }
    ).hide(axis="index")
)


In [ ]:
corpus_costs = (
    results_df.groupby("source", as_index=False)
    .agg(
        successful_calls=("snapshot_file_name", "size"),
        total_api_spend_usd=("cost_usd", "sum"),
        average_cost_per_api_call_usd=("cost_usd", "mean"),
        input_tokens=("input_tokens", "sum"),
        cached_input_tokens=("cached_input_tokens", "sum"),
        output_tokens=("output_tokens", "sum"),
    )
    .sort_values("source")
)
display(
    corpus_costs.style.format(
        {
            "total_api_spend_usd": "${:,.4f}",
            "average_cost_per_api_call_usd": "${:,.4f}",
            "input_tokens": "{:,}",
            "cached_input_tokens": "{:,}",
            "output_tokens": "{:,}",
        }
    ).hide(axis="index")
)


## Validation summary

These counts describe model-generated screening observations and candidate proposals. They are evidence for human adjudication, not independent votes or automatic schema-revision decisions.

In [ ]:
fit_order = ["covered", "weak_fit", "no_fit", "uncertain", "out_of_scope"]
fit_status_summary = (
    observations_df["fit_status"]
    .value_counts()
    .reindex(fit_order, fill_value=0)
    .rename_axis("fit_status")
    .reset_index(name="observations")
)
fit_status_summary["percentage"] = (
    100 * fit_status_summary["observations"] / fit_status_summary["observations"].sum()
)

candidate_level_counts = candidates_df["source_level"].value_counts()
top_candidate_counts = candidates_df["proposed_name"].value_counts().head(5)
top_candidate_text = ", ".join(
    f"`{name}` ({count})" for name, count in top_candidate_counts.items()
)
covered_percentage = 100 * (observations_df["fit_status"] == "covered").mean()
document_candidate_percentage = (
    100 * candidate_level_counts.get("document", 0) / len(candidates_df)
)
snapshots_with_candidates = candidates_df["snapshot_file_name"].nunique()

display(
    Markdown(
        f"""**Short summary.** All **{len(results_df):,}** snapshots completed successfully, with **{len(unresolved_errors):,}** unresolved errors. The model generated **{len(observations_df):,}** observations; **{covered_percentage:.1f}%** were classified as covered by v1.1. It proposed **{len(candidates_df):,}** candidate fields across **{snapshots_with_candidates:,}** snapshots. **{document_candidate_percentage:.1f}%** of proposals were document-level, so the strongest recurring signal concerns parent-document provenance rather than snapshot content alone. The most frequent exact proposed names were {top_candidate_text}.

Exact names remain unconsolidated for human review."""
    )
)
display(
    fit_status_summary.style.format(
        {"observations": "{:,}", "percentage": "{:.1f}%"}
    ).hide(axis="index")
)

corpus_summary_rows = []
for source, source_results in results_df.groupby("source", sort=True):
    source_observations = observations_df[observations_df["source"] == source]
    source_candidates = candidates_df[candidates_df["source"] == source]
    source_fit_counts = source_observations["fit_status"].value_counts()
    corpus_summary_rows.append(
        {
            "source": source,
            "snapshots": len(source_results),
            "figures": (source_results["artifact_type"] == "figure").sum(),
            "tables": (source_results["artifact_type"] == "table").sum(),
            "observations": len(source_observations),
            "covered_percentage": 100
            * source_fit_counts.get("covered", 0)
            / len(source_observations),
            "weak_fit": source_fit_counts.get("weak_fit", 0),
            "no_fit": source_fit_counts.get("no_fit", 0),
            "candidate_proposals": len(source_candidates),
            "document_candidate_percentage": 100
            * (source_candidates["source_level"] == "document").mean(),
        }
    )

corpus_validation_summary = pd.DataFrame(corpus_summary_rows)
display(
    corpus_validation_summary.style.format(
        {
            "snapshots": "{:,}",
            "figures": "{:,}",
            "tables": "{:,}",
            "observations": "{:,}",
            "covered_percentage": "{:.1f}%",
            "weak_fit": "{:,}",
            "no_fit": "{:,}",
            "candidate_proposals": "{:,}",
            "document_candidate_percentage": "{:.1f}%",
        }
    ).hide(axis="index")
)


## Candidate-field review

The table ranks exact proposed names by recurrence. Similar names may represent the same underlying concept and should be consolidated only during human review.

In [ ]:
candidate_summary = (
    candidates_df.groupby("proposed_name", as_index=False)
    .agg(
        proposals=("snapshot_file_name", "size"),
        snapshots=("snapshot_file_name", "nunique"),
        sources=("source", lambda values: ", ".join(sorted(set(values)))),
        source_levels=("source_level", lambda values: ", ".join(sorted(set(values)))),
        weak_fit_support=("weak_fit_support", "sum"),
        no_fit_support=("no_fit_support", "sum"),
    )
    .sort_values(["proposals", "proposed_name"], ascending=[False, True])
    .reset_index(drop=True)
)
display(candidate_summary.head(30).style.hide(axis="index"))
print(
    f"Showing the 30 most frequent of {len(candidate_summary):,} exact names; "
    f"{(candidate_summary['proposals'] == 1).sum():,} names occurred once."
)


### Interpretation and human-review priorities

In [ ]:
candidate_source_counts = (
    candidates_df.groupby("proposed_name")["source"].nunique().rename("corpora")
)
cross_corpus_candidates = (
    candidate_summary.merge(candidate_source_counts, on="proposed_name")
    .query("corpora == @results_df.source.nunique()")
    .head(5)
)
cross_corpus_text = ", ".join(
    f"`{row.proposed_name}` ({row.proposals})"
    for row in cross_corpus_candidates.itertuples()
)

snapshot_candidate_counts = (
    candidates_df[candidates_df["source_level"].isin(["snapshot", "both"])][
        "proposed_name"
    ]
    .value_counts()
    .head(5)
)
snapshot_candidate_text = ", ".join(
    f"`{name}` ({count})" for name, count in snapshot_candidate_counts.items()
)

covered_minimum = corpus_validation_summary["covered_percentage"].min()
covered_maximum = corpus_validation_summary["covered_percentage"].max()
document_candidates = candidate_level_counts.get("document", 0)
snapshot_candidates = candidate_level_counts.get("snapshot", 0)
both_level_candidates = candidate_level_counts.get("both", 0)
singleton_candidates = (candidate_summary["proposals"] == 1).sum()
unhcr_document_percentage = corpus_validation_summary.loc[
    corpus_validation_summary["source"] == "unhcr",
    "document_candidate_percentage",
].iloc[0]

display(
    Markdown(
        f"""1. **Coverage is broadly consistent across corpora.** The covered share ranges from **{covered_minimum:.1f}% to {covered_maximum:.1f}%**, suggesting that v1.1 captures most commonly observed snapshot-description concepts across the three sources. `weak_fit` and `no_fit` observations remain screening signals for review rather than automatic schema failures.

2. **Parent-document provenance is the dominant gap family.** Of **{len(candidates_df):,}** proposals, **{document_candidates:,}** are document-level, compared with **{snapshot_candidates:,}** snapshot-level and **{both_level_candidates:,}** supported by both levels. Exact candidates recurring in all three corpora include {cross_corpus_text}. This pattern supports reviewing whether v1.1 needs additional provenance fields or more explicit definitions and scope guidance.

3. **Candidate frequency is affected by the supplied corpus metadata.** UNHCR has **{unhcr_document_percentage:.1f}%** document-level proposals. Recurrence therefore reflects both schema coverage and which document attributes each corpus exposes; counts should not be treated as independent votes.

4. **Recurring snapshot-level signals are narrower.** The most frequent exact names supported by snapshot evidence are {snapshot_candidate_text}. These should be reviewed separately from parent-document administration so that systematic provenance gaps do not obscure possible snapshot-content gaps.

5. **Human normalization is essential.** The run produced **{len(candidate_summary):,}** exact names, including **{singleton_candidates:,}** singletons. Likely aliases such as URL/locator and creator/author/contributor/producer should be compared by definition and evidence, while preserving genuinely different semantic roles.

**Suggested review sequence:** (a) normalize candidate families without deciding acceptance; (b) adjudicate cross-corpus provenance concepts as new fields versus documentation clarifications; (c) review recurring snapshot-level concepts for operational value; and (d) inspect singleton evidence qualitatively before rejecting it for low frequency."""
    )
)


### Inspect evidence for one candidate

Change `CANDIDATE_NAME` or `EVIDENCE_LIMIT` and rerun the cell. Evidence is sampled across source corpora when possible; the full supporting rows remain available in `candidate_evidence_df`.

In [ ]:
def inspect_candidate(proposed_name: str, max_evidence_rows: int = 12) -> None:
    """Display definitions and supporting evidence for one proposed name.

    Parameters
    ----------
    proposed_name : str
        Exact candidate name to inspect.
    max_evidence_rows : int, optional
        Maximum number of supporting observations to display. Defaults to 12.

    Raises
    ------
    KeyError
        If the exact proposed name is not present.
    ValueError
        If ``max_evidence_rows`` is less than one.
    """
    if max_evidence_rows < 1:
        raise ValueError("max_evidence_rows must be at least one.")
    matching_candidates = candidates_df[candidates_df["proposed_name"] == proposed_name]
    if matching_candidates.empty:
        raise KeyError(f"Unknown candidate name: {proposed_name}")

    display(Markdown(f"#### `{proposed_name}`"))
    candidate_details = (
        matching_candidates[
            [
                "definition",
                "why_existing_fields_are_insufficient",
                "operational_value",
                "source_level",
            ]
        ]
        .drop_duplicates()
        .head(5)
    )
    display(
        candidate_details.style.set_properties(
            **{"white-space": "pre-wrap", "text-align": "left"}
        ).hide(axis="index")
    )

    evidence = candidate_evidence_df[
        candidate_evidence_df["proposed_name"] == proposed_name
    ].sort_values(["source", "snapshot_file_name", "observation_id"])
    rows_per_source = max(1, max_evidence_rows // evidence["source"].nunique())
    evidence_sample = (
        evidence.groupby("source", sort=True, group_keys=False)
        .head(rows_per_source)
        .head(max_evidence_rows)
    )
    display(
        evidence_sample[
            [
                "snapshot_file_name",
                "source",
                "artifact_type",
                "fit_status",
                "metadata_concept",
                "evidence_source",
                "evidence",
                "fit_rationale",
            ]
        ]
        .style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})
        .hide(axis="index")
    )
    print(
        f"Showing {len(evidence_sample):,} of {len(evidence):,} supporting observations."
    )


CANDIDATE_NAME = candidate_summary.iloc[0]["proposed_name"]
EVIDENCE_LIMIT = 12
inspect_candidate(CANDIDATE_NAME, EVIDENCE_LIMIT)


## Stability assessment: were the original 210 snapshots enough?

The held-out run tests whether v1.1 continues to cover common snapshot concepts and whether it exposes recurring, operationally meaningful gaps. Stability here means that new evidence is concentrated in a bounded set of reviewable concepts; it does not mean that no future snapshot can produce a new singleton.

In [ ]:
candidate_scope_rows = []
candidate_scopes = {
    "document": {"document"},
    "snapshot or both": {"snapshot", "both"},
}
for scope_name, source_levels in candidate_scopes.items():
    scope_candidates = candidates_df[candidates_df["source_level"].isin(source_levels)]
    exact_name_counts = scope_candidates["proposed_name"].value_counts()
    candidate_scope_rows.append(
        {
            "candidate_scope": scope_name,
            "proposals": len(scope_candidates),
            "affected_snapshots": scope_candidates["snapshot_file_name"].nunique(),
            "exact_names": len(exact_name_counts),
            "singleton_names": (exact_name_counts == 1).sum(),
            "recurring_names": (exact_name_counts >= 2).sum(),
            "weak_fit_support": scope_candidates["weak_fit_support"].sum(),
            "no_fit_support": scope_candidates["no_fit_support"].sum(),
        }
    )

candidate_scope_summary = pd.DataFrame(candidate_scope_rows)
display(candidate_scope_summary.style.hide(axis="index"))

noncovered_by_evidence = pd.crosstab(
    observations_df["evidence_source"], observations_df["fit_status"]
).reindex(
    index=["snapshot", "document", "both"],
    columns=["weak_fit", "no_fit"],
    fill_value=0,
)
display(noncovered_by_evidence.style.format("{:,}"))


**Observation / decision.** The core snapshot schema is **provisionally stable, but not saturated**. Only 75 of 202 held-out snapshots produced any snapshot-level candidate, and those 126 proposals contained just 24 `no_fit` supports; most were `weak_fit`, indicating a nearby existing field or a documentation problem rather than an unambiguous missing field. The 83 exact snapshot-level names also have a long tail of 64 singletons.

Parent-document provenance is the important exception: 193 snapshots produced 452 document-level proposals, including 369 `no_fit` supports concentrated in a small recurring set. Thus, the original 210 snapshots appear sufficient to establish the **common snapshot-content core**, but v1.1 still needs human adjudication of a bounded provenance and role-specific agenda. Additional broad sampling is unlikely to be more informative than resolving these recurring families first. This is a stability judgment, not a claim of universal schema saturation.

### Why retain `covered` observations?

`covered` observations are excluded from candidate-field adjudication, but they should not be deleted from the analytical record. They provide the denominator for the 79.1% descriptive coverage result, demonstrate that the model actually inspected common concepts rather than merely failing to mention them, permit corpus-level comparisons, and preserve positive evidence for which v1.1 fields are working. Asking only for gaps would encourage gap invention and make silence ambiguous. The final gap-review tables therefore filter out `covered`, while the raw JSONL and aggregate coverage tables retain it for auditability.

### Consequential candidate families

The following provisional families consolidate obvious naming variants only for stability screening. They are not adjudicated schema fields.

In [ ]:
CANDIDATE_FAMILIES = {
    "Source-document retrieval": {
        "source_document_url",
        "source_document_locator",
    },
    "Source-document identity and type": {
        "source_document_identifier",
        "source_document_type",
    },
    "Source-document publication date": {
        "source_document_publication_date",
        "source_document_date",
    },
    "Source-document attribution": {
        "source_document_creator",
        "source_document_contributor",
        "source_document_author",
        "source_document_producer",
        "source_document_organization",
        "source_document_attribution",
    },
    "Snapshot artifact date": {
        "snapshot_date",
        "snapshot_production_date",
        "snapshot_creation_date",
    },
    "Snapshot artifact attribution": {
        "snapshot_creator",
        "snapshot_producer",
        "snapshot_contributor",
        "snapshot_attribution",
    },
    "Cartographic scale and coordinates": {
        "map_scale",
        "cartographic_scale",
        "map_scale_unit",
        "coordinate_grid",
        "coordinate_extent",
    },
    "Analytical variable role": {
        "axis_variable",
        "dependent_variable",
        "explanatory_variable",
        "outcome_variable",
        "model_variable_role",
    },
    "Classification semantics": {
        "classification_system",
        "classification_level",
        "classification_rule",
        "category_hierarchy",
    },
    "Composite visualization structure": {
        "visualization_composition",
        "visualization_component_type",
        "visualization_components",
        "embedded_visualization_type",
    },
    "Data-collection timing and responsibility": {
        "data_collection_frequency",
        "data_collection_schedule",
        "data_collection_responsibility",
        "data_collection_responsible_party",
    },
    "Sample size": {"sample_size"},
    "Uncertainty representation": {"uncertainty_representation"},
    "Reference event or context": {
        "reference_event",
        "temporal_reference_event",
        "event_context",
    },
}

family_rows = []
for family, proposed_names in CANDIDATE_FAMILIES.items():
    family_candidates = candidates_df[
        candidates_df["proposed_name"].isin(proposed_names)
    ]
    family_rows.append(
        {
            "review_family": family,
            "proposals": len(family_candidates),
            "snapshots": family_candidates["snapshot_file_name"].nunique(),
            "sources": ", ".join(sorted(family_candidates["source"].unique())),
            "source_levels": ", ".join(
                sorted(family_candidates["source_level"].unique())
            ),
            "weak_fit_support": family_candidates["weak_fit_support"].sum(),
            "no_fit_support": family_candidates["no_fit_support"].sum(),
            "exact_names": ", ".join(
                sorted(family_candidates["proposed_name"].unique())
            ),
        }
    )

candidate_family_summary = (
    pd.DataFrame(family_rows)
    .sort_values(["proposals", "review_family"], ascending=[False, True])
    .reset_index(drop=True)
)
display(
    candidate_family_summary.style.set_properties(
        subset=["review_family", "exact_names"],
        **{"white-space": "pre-wrap", "text-align": "left"},
    ).hide(axis="index")
)


**Observation / comments.** Source-document retrieval, identity/type, publication date, and attribution are the only large, systematic families and are the main challenge to stability as currently documented. Among snapshot-level concepts, artifact dates and creators recur mostly in older World Bank maps; cartographic scale/coordinates are a coherent specialized family; and analytical variable roles and classification semantics recur primarily in PRWP statistical artifacts. Those latter patterns may call for clearer definitions or structured role qualifiers rather than many independent new fields. Sample size and uncertainty representation have low frequency but high generalizability, so they warrant qualitative review despite only two proposals each.

In [ ]:
family_by_candidate = {
    proposed_name: family
    for family, proposed_names in CANDIDATE_FAMILIES.items()
    for proposed_name in proposed_names
}
consequential_evidence = candidate_evidence_df.copy()
consequential_evidence["review_family"] = consequential_evidence["proposed_name"].map(
    family_by_candidate
)
consequential_evidence = consequential_evidence.dropna(subset=["review_family"])
consequential_evidence["fit_priority"] = consequential_evidence["fit_status"].map(
    {"no_fit": 0, "weak_fit": 1}
)
consequential_evidence_sample = (
    consequential_evidence.sort_values(
        [
            "review_family",
            "fit_priority",
            "source",
            "snapshot_file_name",
        ]
    )
    .groupby(["review_family", "fit_status"], sort=False, group_keys=False)
    .head(1)
)
display(
    consequential_evidence_sample[
        [
            "review_family",
            "proposed_name",
            "fit_status",
            "source",
            "snapshot_file_name",
            "metadata_concept",
            "evidence",
            "fit_rationale",
        ]
    ]
    .style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})
    .hide(axis="index")
)


**Weak-fit and no-fit comments.** `weak_fit` is analytically important because it distinguishes likely schema-definition or granularity problems from clear absence. Consequential weak-fit examples include map scale being forced toward `unit_of_measure`, statistical variable roles being forced into `variable_name`, and classification standards being relegated to `interpretive_note`. Consequential snapshot-level `no_fit` examples include explicit artifact creators/dates, coordinate grids, data-collection responsibility, sample size, and uncertainty markers. One-off proposals such as `axis_assignment`, `contact_point`, and `visual_annotation` are interesting but do not by themselves challenge overall stability.

In [ ]:
boundary_observations = (
    observations_df[observations_df["fit_status"].isin(["out_of_scope", "uncertain"])]
    .sort_values(["fit_status", "source", "snapshot_file_name"])
    .reset_index(drop=True)
)
display(
    boundary_observations[
        [
            "snapshot_file_name",
            "source",
            "artifact_type",
            "fit_status",
            "metadata_concept",
            "evidence_source",
            "evidence",
            "fit_rationale",
        ]
    ]
    .style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})
    .hide(axis="index")
)


**Out-of-scope and uncertain comments.** The three `out_of_scope` rows are regression estimates and diagnostics presented as table contents; they confirm the intended boundary and do not indicate a schema gap. The 20 `uncertain` rows are also substantively useful: most involve unlabeled footer/report dates, document-title dates whose applicability to the snapshot is unclear, unidentified map variables or boundary levels, icon-only categories, or ambiguous organizational attribution. These are evidence-quality and semantic-role ambiguities, not recurring missing-field evidence. Their small number supports retaining `uncertain` as an audit category without treating it as a candidate source.

## Results overview

In [ ]:
overview_columns = [
    "snapshot_file_name",
    "source",
    "artifact_type",
    "service_tier",
    "observations",
    "covered",
    "weak_fit",
    "no_fit",
    "out_of_scope",
    "uncertain",
    "candidate_fields",
    "cost_usd",
]
display(
    results_df[overview_columns]
    .style.format({"cost_usd": "${:,.4f}"})
    .set_properties(subset=["snapshot_file_name"], **{"text-align": "left"})
)


## Unresolved errors

In [ ]:
if unresolved_errors:
    error_columns = [
        "snapshot_file_name",
        "source",
        "artifact_type",
        "error_stage",
        "error_type",
        "error",
    ]
    display(
        pd.DataFrame(unresolved_errors)[error_columns].style.set_properties(
            **{"white-space": "pre-wrap", "text-align": "left"}
        )
    )
else:
    display(Markdown("_No unresolved errors._"))


## Inspect one result

Change `SNAPSHOT_INDEX` to any row index shown in the results overview.

In [ ]:
def inspect_result(snapshot_file_name: str) -> None:
    """Display the image and structured assessment for one snapshot.

    Parameters
    ----------
    snapshot_file_name : str
        Exact successful snapshot filename to inspect.

    Raises
    ------
    KeyError
        If no successful result exists for the filename.
    """
    record = records_by_name[snapshot_file_name]
    parsed = record["parsed_output"]
    summary = results_df.loc[results_df["snapshot_file_name"] == snapshot_file_name]

    snapshot_matches = list(SNAPSHOTS_DIR.glob(f"*/*/{snapshot_file_name}"))
    if len(snapshot_matches) != 1:
        raise ValueError(
            f"Expected one snapshot for {snapshot_file_name!r}, "
            f"found {len(snapshot_matches)}."
        )

    display(Markdown(f"### `{snapshot_file_name}`"))
    display(Image(filename=str(snapshot_matches[0]), width=900))
    display(
        summary[
            [
                "source",
                "artifact_type",
                "model",
                "service_tier",
                "elapsed_seconds",
                "input_tokens",
                "cached_input_tokens",
                "output_tokens",
                "cost_usd",
            ]
        ]
        .style.format(
            {
                "elapsed_seconds": "{:,.1f}",
                "input_tokens": "{:,}",
                "cached_input_tokens": "{:,}",
                "output_tokens": "{:,}",
                "cost_usd": "${:,.4f}",
            }
        )
        .hide(axis="index")
    )

    observations = pd.DataFrame(parsed["observations"])
    observations["closest_schema_fields"] = observations[
        "closest_schema_fields"
    ].str.join(", ")
    display(Markdown("#### Observations"))
    display(
        observations[
            [
                "observation_id",
                "fit_status",
                "metadata_concept",
                "closest_schema_fields",
                "evidence_source",
                "evidence",
                "fit_rationale",
            ]
        ].style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})
    )

    candidates = pd.DataFrame(parsed["candidate_new_fields"])
    display(Markdown("#### Candidate fields"))
    if candidates.empty:
        display(Markdown("_No candidate fields proposed._"))
    else:
        candidates["supporting_observation_ids"] = candidates[
            "supporting_observation_ids"
        ].str.join(", ")
        display(
            candidates.style.set_properties(
                **{"white-space": "pre-wrap", "text-align": "left"}
            )
        )


In [ ]:
SNAPSHOT_INDEX = 0
inspect_result(results_df.iloc[SNAPSHOT_INDEX]["snapshot_file_name"])
